# Предобработка данных TAIGA: gamma vs proton

Цель работы: подготовить данные Хилласа к обучению моделей классификации частиц: `gamma` или `proton`.

В этой части работы нужно:

1. Считать CSV-файлы с данными.
2. Проверить структуру и качество данных.
3. Построить распределения признаков для классов `gamma` и `proton`.
4. Исследовать связь признаков с целевым классом.
5. Сформировать итоговый набор признаков для последующего обучения моделей.

## Словарь признаков для текущих CSV

Ниже приведена рабочая расшифровка признаков, которые есть в текущих файлах `data/gamma.csv` и `data/proton.csv` и соотносятся с PDF `Parameters_Definitions_Report.pdf`.

Важное ограничение: PDF описывает более полные экспериментальные таблицы TAIGA-IACT02, а текущие CSV являются сокращёнными таблицами признаков для задачи `gamma` vs `proton`. Поэтому информация из PDF подходит к нашим данным только частично. Хорошо применимы разделы про Hillas-параметры, моменты формы изображения и параметры клининга. В текущих CSV нет многих полей наблюдений, времени, pointing-файлов, фона и темпа счёта: `por`, `time`, `unix_time_ns`, `delta_time`, `error_deg`, `tel_az`, `tel_el`, `source_az`, `source_el`, `CR400phe`, `CR_portion`, `median_background`, `MAD_background`, `tel_ra`, `tel_dec`, `source_ra`, `source_dec`, `tracking`, `good`, `star`, `weather_mark`, `alpha_c`. Поэтому в этой работе используйте только фактически присутствующие колонки.

| Колонка в текущих CSV | Описание |
|---|---|
| `event_number` | Номер события. В PDF: `event_numb`. |
| `numb_pix` | Число пикселей, прошедших клининг. |
| `num_islands` | Число островов в событии после клининга. |
| `size` | Суммарное число фотоэлектронов в событии. |
| `Xc[0]` | X-координата центра тяжести события. |
| `Yc[0]` | Y-координата центра тяжести события. |
| `con_selected_island` | Доля `size` в отобранном самом ярком острове. |
| `con2` | Доля `size` в двух самых ярких пикселях события. |
| `length[0]` | Момент второго порядка вдоль главной оси эллипса. |
| `width[0]` | Момент второго порядка вдоль малой оси эллипса. |
| `dist[0]` | Расстояние между центром тяжести эллипса и положением источника. В PDF: `dist1`. |
| `dist[1]` | Расстояние между центром тяжести эллипса и положением антиисточника. В PDF: `dist2`. |
| `a_axis[0]` | Коэффициент `a` уравнения `y = a*x + b`, характеризующий наклон эллипса относительно координатной сетки камеры. |
| `b_axis[0]` | Коэффициент `b` уравнения `y = a*x + b`. Этого поля нет в исходной таблице описаний, но оно согласуется с описанием `a_axis`. |
| `skewness_l` | Асимметрия события вдоль главной оси эллипса. |
| `skewness_w` | Асимметрия события вдоль малой оси эллипса. |
| `skewness` | Асимметрия события. При анализе нужно проверить, как этот вариант названия соотносится с моментами вдоль осей. |
| `kurtosis_l` | Эксцесс вдоль главной оси. |
| `kurtosis_w` | Эксцесс вдоль малой оси. |
| `kurtosis` | Эксцесс события. При анализе нужно проверить, как этот вариант названия соотносится с моментами вдоль осей. |
| `alpha[0]` | Угол между главной осью эллипса и направлением на источник. В PDF: `alpha1`. |
| `alpha[1]` | Угол между главной осью эллипса и направлением на антиисточник. В PDF: `alpha2`. |
| `source_x` | X-координата источника в камере в градусах. |
| `source_y` | Y-координата источника в камере в градусах. |
| `edge` | Индикатор наличия граничных пикселей камеры среди пикселей, прошедших клининг. |

Колонки без надежного описания из предоставленного словаря, но присутствующие в данных: `clean_number`, `axis_scatter`, `energy`, `azwidth[1]`, `miss[1]`, `x_max_coord`, `y_max_coord`, `x_ground`, `y_ground`, `tel_tet`, `tel_fi`, `source_tet`, `source_fi`, `Xmax`, `con1`, `con3`.

Для итогового выбора признаков важно не только название, но и проверка распределений, пропусков, корреляций и возможной утечки информации.

## Как выполнять работу

- Выполняйте задания строго по порядку.
- В кодовых ячейках даны только подсказки по нужным методам и направлению решения. Рабочий код нужно написать самостоятельно.
- После каждого задания заполните блок `Промежуточный вывод`.
- Не ограничивайтесь фразой "видно на графике". Пишите, какие именно признаки отличаются и почему это важно для будущей модели.
- В конце работы должен получиться список признаков `selected_features`, который можно использовать в следующем ноутбуке для обучения моделей.

## Подготовка окружения

Импортируйте библиотеки, которые понадобятся для анализа таблиц и визуализации.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

sns.set_theme(style="whitegrid", context="notebook")
RANDOM_STATE = 42

## Задание 1. Считать CSV-данные

В папке `data` лежат два подготовленных CSV-файла:

- `gamma.csv` - события класса gamma;
- `proton.csv` - события класса proton.

Считайте оба файла в pandas DataFrame и посмотрите первые строки.

In [ ]:
# Подсказки:
# 1. Создайте переменные с путями к файлам: data/gamma.csv и data/proton.csv.
# 2. Считайте оба файла с помощью pandas: pd.read_csv(...).
# 3. Сохраните таблицы в переменные gamma и proton.
# 4. Выведите размер таблиц через .shape.
# 5. Посмотрите первые строки через .head() или display(...).

# Ваш код:


### Промежуточный вывод 1

Заполните после выполнения кода:

- Сколько строк и колонок в каждом файле?
- Какие первые признаки вы видите в таблицах?
- Есть ли признаки, которые уже по названию выглядят как служебные или идентификаторы событий?

**Ваш вывод:**

> ...

## Задание 2. Объединить данные и добавить метку класса

Объедините данные `gamma` и `proton` в одну таблицу. Перед объединением добавьте информацию о классе события:

- `target = 0` для `gamma`;
- `target = 1` для `proton`.

После объединения сохраните результат в переменную `df`.

In [ ]:
# Подсказки:
# 1. Сделайте копии таблиц gamma и proton через .copy().
# 2. Добавьте в каждую копию числовую колонку target.
# 3. Объедините таблицы через pd.concat(..., ignore_index=True).
# 4. Проверьте размер df и посмотрите несколько строк.

# Ваш код:


### Промежуточный вывод 2

Ответьте:

- Сколько строк получилось после объединения?
- Что означает значение `target = 0` и `target = 1`?
- Что вы заметили при первом просмотре объединённой таблицы?

**Ваш вывод:**

> ...

## Задание 3. Осмотреть структуру объединённой таблицы

Изучите структуру `df`: размеры, названия колонок, типы данных и несколько случайных строк. На этом этапе не исправляйте данные, а только зафиксируйте наблюдения.

In [ ]:
# Подсказки:
# 1. Используйте .shape, .columns, .dtypes или .info().
# 2. Посмотрите первые, последние и случайные строки через .head(), .tail(), .sample().
# 3. Выпишите наблюдения, которые могут быть важны для дальнейшей предобработки.

# Ваш код:


### Промежуточный вывод 3

Ответьте:

- Какие типы данных есть в таблице?
- Какие колонки выглядят физическими признаками, а какие служебными?
- Какие особенности структуры таблицы нужно учесть перед выбором признаков?

**Ваш вывод:**

> ...

## Задание 4. Проверить качество таблицы

Проверьте:

- баланс классов;
- типы данных;
- пропуски;
- полные дубликаты строк.

Эти проверки нужны до построения распределений и выбора признаков.

In [ ]:
# Подсказки:
# 1. Посчитайте число объектов каждого класса через value_counts().
# 2. Посчитайте доли классов через value_counts(normalize=True).
# 3. Проверьте типы данных через .dtypes.
# 4. Найдите долю пропусков по колонкам через .isna().mean().
# 5. Проверьте полные дубликаты через .duplicated().sum().

# Ваш код:


### Промежуточный вывод 4

Ответьте:

- Сильно ли отличаются размеры классов?
- Есть ли пропуски? Если есть, в каких колонках?
- Есть ли проблема с типами данных?
- Нужно ли сейчас удалять дубликаты или достаточно зафиксировать их наличие?

**Ваш вывод:**

> ...

## Задание 5. Выбрать числовые признаки для первичного анализа

Не все числовые колонки должны становиться признаками модели. Например, `target` - это целевая переменная, а `event_number` и `clean_number` похожи на идентификаторы.

На этом шаге сформируйте список числовых колонок, которые можно анализировать как потенциальные признаки.

In [ ]:
# Подсказки:
# 1. Составьте список технических колонок, которые не должны быть признаками модели.
#    Например: target, event_number, clean_number.
# 2. Получите числовые колонки через select_dtypes(include="number").
# 3. Исключите технические колонки из списка числовых.
# 4. Сохраните результат в переменную numeric_columns.
# 5. Изучите описательные статистики через describe().

# Ваш код:


### Промежуточный вывод 5

Ответьте:

- Какие колонки вы исключили как технические?
- Все ли оставшиеся числовые колонки физически осмысленны как признаки?
- Есть ли признаки с очень разными масштабами значений?

**Ваш вывод:**

> ...

## Задание 6. Построить распределения признаков

Постройте распределения признаков отдельно для классов `gamma` и `proton`.

Начните с нескольких признаков, которые встречались в исследовательском ноутбуке: `width[0]`, `length[0]`, `alpha[0]`, `miss[1]`, `azwidth[1]`, `dist[1]`, `Xc[0]`, `Yc[0]`, `numb_pix`.

Затем добавьте 2-3 признака по своему выбору.

In [ ]:
# Подсказки:
# 1. Создайте список features_to_plot с несколькими признаками для анализа.
#    Можно начать с width[0], length[0], alpha[0], miss[1], azwidth[1], dist[1], Xc[0], Yc[0], numb_pix.
# 2. Для каждого признака постройте распределение отдельно для gamma и proton.
# 3. Удобные функции: sns.histplot(..., hue="target") или sns.kdeplot(..., hue="target").
# 4. Настройте bins, alpha, stat="density" и common_norm=False, если используете histplot.
# 5. Подпишите графики и сравните формы распределений.

# Ваш код:


In [ ]:
# Подсказки:
# 1. Постройте boxplot или violinplot для тех же признаков.
# 2. Удобные функции: sns.boxplot(data=df, x="target", y=feature) или sns.violinplot(...).
# 3. Используйте цикл по features_to_plot, если хотите построить несколько графиков.
# 4. Обратите внимание на медианы, разброс и выбросы.

# Ваш код:


### Промежуточный вывод 6

Ответьте:

- Какие признаки имеют заметно разные распределения для `gamma` и `proton`?
- Какие признаки почти не разделяют классы визуально?
- Есть ли признаки с сильными выбросами?
- Какие признаки стоит оставить кандидатами для модели по итогам визуального анализа?

**Ваш вывод:**

> ...

## Задание 7. Исследовать корреляцию признаков с таргетом

Так как `target` закодирован числами `0` и `1`, обычная корреляция Пирсона между числовым признаком и `target` показывает линейную связь признака с классом.

Интерпретация:

- положительная корреляция: большие значения признака чаще соответствуют `proton`;
- отрицательная корреляция: большие значения признака чаще соответствуют `gamma`;
- корреляция около нуля не означает, что признак бесполезен: связь может быть нелинейной.

In [ ]:
# Подсказки:
# 1. Посчитайте корреляцию каждого числового признака с df["target"].
# 2. Удобный метод: .corrwith(...).
# 3. Добавьте колонку с модулем корреляции, чтобы отсортировать признаки по силе связи.
# 4. Сохраните таблицу корреляций в переменную corr_with_target.
# 5. Выведите признаки с самой большой корреляцией по модулю.

# Ваш код:


In [ ]:
# Подсказки:
# 1. Возьмите top-N признаков из corr_with_target.
# 2. Постройте горизонтальный barplot корреляций.
# 3. Удобная функция: sns.barplot(...).
# 4. Добавьте вертикальную линию x=0 через plt.axvline(0).
# 5. Подпишите оси и заголовок.

# Ваш код:


### Промежуточный вывод 7

Ответьте:

- Какие признаки сильнее всего связаны с `target` по модулю корреляции?
- Какие признаки имеют положительную корреляцию с `target`, то есть больше характерны для `proton`?
- Какие признаки имеют отрицательную корреляцию с `target`, то есть больше характерны для `gamma`?
- Совпадают ли результаты корреляции с выводами по распределениям?

**Ваш вывод:**

> ...

## Задание 8. Проверить корреляции между признаками

Если два признака почти полностью повторяют друг друга, модель может не получить от них новой информации. Постройте тепловую карту корреляций между наиболее перспективными признаками.

Сначала выберите признаки с наибольшей связью с `target`, затем посмотрите, нет ли среди них сильно коррелирующих пар.

In [ ]:
# Подсказки:
# 1. Выберите 10-15 наиболее перспективных признаков.
# 2. Посчитайте матрицу корреляций между ними через .corr().
# 3. Постройте heatmap через sns.heatmap(...).
# 4. Найдите пары признаков с очень высокой корреляцией между собой.

# Ваш код:


### Промежуточный вывод 8

Ответьте:

- Есть ли пары признаков с очень высокой корреляцией между собой?
- Нужно ли удалять один из таких признаков уже сейчас или лучше оставить решение до этапа обучения моделей?
- Какие признаки после этого шага остаются наиболее перспективными?

**Ваш вывод:**

> ...

## Задание 9. Сформировать итоговый набор признаков

Соберите итоговый список `selected_features` для следующей работы с моделями.

Критерии выбора:

- признак есть и у `gamma`, и у `proton`;
- признак числовой;
- признак не является `target` или идентификатором события;
- распределения признака отличаются между классами или есть заметная связь с `target`;
- если признак исключен, вы можете объяснить почему.

In [ ]:
# Подсказки:
# 1. Создайте список selected_features вручную по итогам предыдущих заданий.
# 2. В список должны попасть только признаки, которые есть в df и подходят для модели.
# 3. Не включайте target и идентификаторы событий.
# 4. Проверьте список: все выбранные признаки должны быть в numeric_columns.
# 5. Выведите итоговый список и его длину.

# Ваш код:
selected_features = [
    # впишите выбранные признаки здесь
]


## Итоговый вывод

Заполните в конце работы:

1. Какие файлы были считаны и как был задан `target`?
2. Какие признаки визуально лучше всего разделяют `gamma` и `proton`?
3. Какие признаки сильнее всего коррелируют с `target`?
4. Какие признаки были исключены и почему?
5. Какой итоговый список `selected_features` вы предлагаете использовать для обучения моделей?

**Ваш итоговый вывод:**

> ...